In [12]:
import os
import re
import warnings
import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    roc_auc_score,
)
from sklearn.exceptions import ConvergenceWarning

# =========================
# 0. Config
# =========================

MODEL_NAME = "FVs_medsiglip"   # 这里只改一个模型名即可

NPZ_PATH = f"../aligned_features_drvalid/aligned_drvalid_{MODEL_NAME}.npz"
META_PATH = f"../aligned_features_drvalid/aligned_drvalid_{MODEL_NAME}_metadata.csv"

GRADE_COL = "patient_DR_Level"   # 如果你想用眼级别标签，改成 "eye_DR_Level"

N_PC = 50
N_SPLITS = 5
RANDOM_STATE = 42

# 自动剔除标准：
# disease_score = abs(Spearman(PC, DR))
# binary nuisance_score = 2 * abs(AUC_raw - 0.5)
# ordinal nuisance_score = abs(Spearman(PC, nuisance))
# remove if max_nuisance_score > disease_score
#
# 如果你之前没有最低阈值，就保持 0.0
# 如果你想避免剔除很弱的PC，可以改成 0.15 或 0.20
MIN_NUISANCE_SCORE = 0.0

BINARY_NUISANCE_COLS = [
    "eye",
    "view_no",
    "Overall quality",
]

ORDINAL_NUISANCE_COLS = [
    "Clarity",
    "Field definition",
    "Artifact",
]

OUT_DIR = "./kfold_pc_removal_one_model"
os.makedirs(OUT_DIR, exist_ok=True)


# =========================
# 1. Utility functions
# =========================

def add_eye_and_view_from_path(meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    从 image_path 中解析 eye 和 view_no。
    例如:
        xxx/5_l1.jpg -> eye = l, view_no = 1
        xxx/5_r2.jpg -> eye = r, view_no = 2
    如果原本已有 eye/view_no，则不覆盖。
    """
    meta_df = meta_df.copy()

    if "image_path" not in meta_df.columns:
        return meta_df

    path_s = meta_df["image_path"].astype(str)

    if "eye" not in meta_df.columns:
        eye = path_s.str.extract(r"_[lLrR](\d)?(?:\.|$)")[0]
        # 上面只取到了数字，所以重新抽 l/r
        eye = path_s.str.extract(r"_([lLrR])\d?(?:\.|$)")[0]
        meta_df["eye"] = eye.str.lower()

    if "view_no" not in meta_df.columns:
        view_no = path_s.str.extract(r"_[lLrR](\d)(?:\.|$)")[0]
        meta_df["view_no"] = pd.to_numeric(view_no, errors="coerce")

    return meta_df


def safe_abs_spearman(x, y):
    """
    返回:
        abs_rho, raw_rho, p_value
    PCA方向正负是任意的，所以判断强度时用 abs(rho)。
    """
    x = np.asarray(x)
    y = np.asarray(y)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 3:
        return np.nan, np.nan, np.nan

    if len(np.unique(x)) < 2 or len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan

    rho, p = spearmanr(x, y)

    if np.isnan(rho):
        return np.nan, rho, p

    return abs(rho), rho, p


def binary_auc_nuisance_strength(pc_score, factor_values):
    """
    binary nuisance:
        nuisance_score = 2 * abs(AUC_raw - 0.5)

    返回:
        strength, auc_raw, n_valid
    """
    tmp = pd.DataFrame({
        "pc": pc_score,
        "factor": factor_values,
    }).dropna()

    if len(tmp) < 3:
        return np.nan, np.nan, len(tmp)

    unique_vals = sorted(tmp["factor"].unique(), key=lambda v: str(v))

    if len(unique_vals) != 2:
        return np.nan, np.nan, len(tmp)

    mapping = {
        unique_vals[0]: 0,
        unique_vals[1]: 1,
    }

    y_bin = tmp["factor"].map(mapping).astype(int).to_numpy()
    score = tmp["pc"].to_numpy()

    if len(np.unique(y_bin)) < 2:
        return np.nan, np.nan, len(tmp)

    auc_raw = roc_auc_score(y_bin, score)
    strength = 2.0 * abs(auc_raw - 0.5)

    return strength, auc_raw, len(tmp)


def select_pcs_to_remove(
    Z_train,
    meta_train,
    grade_col,
    binary_cols,
    ordinal_cols,
    min_nuisance_score=0.0,
):
    """
    在一个fold的train set里自动选择要剔除的PC。

    Z_train: PCA后的train特征, shape = [n_train, n_pc]
    meta_train: train对应的metadata
    """
    n_pc = Z_train.shape[1]

    y_dr = pd.to_numeric(meta_train[grade_col], errors="coerce").to_numpy()

    rows = []
    remove_indices = []

    for pc_idx in range(n_pc):
        pc_name = f"PC{pc_idx + 1}"
        pc_score = Z_train[:, pc_idx]

        disease_score, disease_rho, disease_p = safe_abs_spearman(pc_score, y_dr)

        nuisance_records = []

        # binary nuisance: eye, view_no, Overall quality 等
        for col in binary_cols:
            if col not in meta_train.columns:
                continue

            strength, auc_raw, n_valid = binary_auc_nuisance_strength(
                pc_score,
                meta_train[col].to_numpy()
            )

            nuisance_records.append({
                "factor": col,
                "factor_type": "binary_auc",
                "nuisance_score": strength,
                "raw_metric": auc_raw,
                "n_valid": n_valid,
            })

        # ordinal nuisance: Clarity, Field definition, Artifact 等
        for col in ordinal_cols:
            if col not in meta_train.columns:
                continue

            y_factor = pd.to_numeric(meta_train[col], errors="coerce").to_numpy()
            strength, rho, p = safe_abs_spearman(pc_score, y_factor)

            nuisance_records.append({
                "factor": col,
                "factor_type": "spearman",
                "nuisance_score": strength,
                "raw_metric": rho,
                "n_valid": np.sum(np.isfinite(y_factor)),
            })

        nuisance_df = pd.DataFrame(nuisance_records)

        if len(nuisance_df) == 0 or nuisance_df["nuisance_score"].dropna().empty:
            max_nuisance_score = np.nan
            max_nuisance_factor = None
            max_nuisance_type = None
            max_nuisance_raw_metric = np.nan
        else:
            best_row = nuisance_df.sort_values(
                "nuisance_score",
                ascending=False,
                na_position="last"
            ).iloc[0]

            max_nuisance_score = best_row["nuisance_score"]
            max_nuisance_factor = best_row["factor"]
            max_nuisance_type = best_row["factor_type"]
            max_nuisance_raw_metric = best_row["raw_metric"]

        # remove = (
        #     np.isfinite(max_nuisance_score)
        #     and np.isfinite(disease_score)
        #     and max_nuisance_score >= min_nuisance_score
        #     and max_nuisance_score > disease_score
        # )
        STRONG_NUISANCE_TH = 0.4
        WEAK_DISEASE_TH = 0.3
        NUISANCE_MARGIN_TH = 0.15

        remove = (
            np.isfinite(max_nuisance_score)
            and np.isfinite(disease_score)
            and max_nuisance_score >= STRONG_NUISANCE_TH
            and disease_score <= WEAK_DISEASE_TH
            and (max_nuisance_score - disease_score) >= NUISANCE_MARGIN_TH
        )        

        if remove:
            remove_indices.append(pc_idx)

        rows.append({
            "PC": pc_name,
            "pc_index": pc_idx,
            "disease_score_abs_spearman": disease_score,
            "disease_spearman_rho": disease_rho,
            "disease_p_value": disease_p,
            "max_nuisance_score": max_nuisance_score,
            "max_nuisance_factor": max_nuisance_factor,
            "max_nuisance_type": max_nuisance_type,
            "max_nuisance_raw_metric": max_nuisance_raw_metric,
            "remove": remove,
        })

    selection_df = pd.DataFrame(rows)
    return remove_indices, selection_df


def evaluate_linear_svm(X_train, y_train, X_valid, y_valid, labels):
    """
    Linear SVM评估。
    这里不额外Standardize PC score，保持PCA空间的尺度。
    如果仍然收敛警告，可以继续增大 max_iter。
    """
    if X_train.shape[1] == 0:
        return {
            "qwk": np.nan,
            "accuracy": np.nan,
            "macro_f1": np.nan,
        }, None

    clf = LinearSVC(
        C=1.0,
        class_weight="balanced",
        dual=False,
        max_iter=100000,
        tol=1e-4,
        random_state=RANDOM_STATE,
    )

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        clf.fit(X_train, y_train)

    pred = clf.predict(X_valid)

    metrics = {
        "qwk": cohen_kappa_score(y_valid, pred, labels=labels, weights="quadratic"),
        "accuracy": accuracy_score(y_valid, pred),
        "macro_f1": f1_score(y_valid, pred, labels=labels, average="macro", zero_division=0),
    }

    return metrics, pred


# =========================
# 2. Load data
# =========================

data = np.load(NPZ_PATH, allow_pickle=True)
X = data["X"]

meta_df = pd.read_csv(META_PATH)
meta_df = add_eye_and_view_from_path(meta_df)

if len(X) != len(meta_df):
    raise ValueError(f"X and metadata length mismatch: X={len(X)}, meta={len(meta_df)}")

if GRADE_COL not in meta_df.columns:
    raise ValueError(f"{GRADE_COL} not found in metadata columns.")

# 去掉DR label缺失行
valid_mask = meta_df[GRADE_COL].notna().to_numpy()

# 去掉特征中含 nan/inf 的行
finite_x_mask = np.isfinite(X).all(axis=1)

mask = valid_mask & finite_x_mask

X = X[mask]
meta_df = meta_df.loc[mask].reset_index(drop=True)

y = pd.to_numeric(meta_df[GRADE_COL], errors="coerce").astype(int).to_numpy()
labels = sorted(np.unique(y))

print(f"Model: {MODEL_NAME}")
print(f"X shape after filtering: {X.shape}")
print("Label distribution:")
print(pd.Series(y).value_counts().sort_index())


# =========================
# 3. K-fold evaluation
# =========================

class_counts = pd.Series(y).value_counts()
actual_n_splits = min(N_SPLITS, int(class_counts.min()))

if actual_n_splits < 2:
    raise ValueError("Not enough samples per class for StratifiedKFold.")

skf = StratifiedKFold(
    n_splits=actual_n_splits,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_rows = []
selection_dfs = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== Fold {fold}/{actual_n_splits} =====")

    X_train_raw = X[train_idx]
    X_valid_raw = X[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    meta_train = meta_df.iloc[train_idx].reset_index(drop=True)
    meta_valid = meta_df.iloc[valid_idx].reset_index(drop=True)

    print(f"train={len(train_idx)}, valid={len(valid_idx)}")

    # 只在train上fit scaler
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train_raw)
    X_valid_std = scaler.transform(X_valid_raw)

    # 只在train上fit PCA
    n_components = min(N_PC, X_train_std.shape[0] - 1, X_train_std.shape[1])

    pca = PCA(
        n_components=n_components,
        random_state=RANDOM_STATE,
    )

    Z_train = pca.fit_transform(X_train_std)
    Z_valid = pca.transform(X_valid_std)

    # 只用train选择要剔除的PC
    remove_indices, selection_df = select_pcs_to_remove(
        Z_train=Z_train,
        meta_train=meta_train,
        grade_col=GRADE_COL,
        binary_cols=BINARY_NUISANCE_COLS,
        ordinal_cols=ORDINAL_NUISANCE_COLS,
        min_nuisance_score=MIN_NUISANCE_SCORE,
    )

    keep_indices = [i for i in range(n_components) if i not in remove_indices]
    removed_pcs = [f"PC{i + 1}" for i in remove_indices]

    print(f"Removed PCs: {removed_pcs if removed_pcs else 'None'}")

    selection_df.insert(0, "fold", fold)
    selection_df.insert(1, "model_name", MODEL_NAME)
    selection_dfs.append(selection_df)

    # baseline: 使用全部前N_PC
    raw_metrics, raw_pred = evaluate_linear_svm(
        Z_train,
        y_train,
        Z_valid,
        y_valid,
        labels=labels,
    )

    # cleaned: 剔除自动选择的PC
    clean_metrics, clean_pred = evaluate_linear_svm(
        Z_train[:, keep_indices],
        y_train,
        Z_valid[:, keep_indices],
        y_valid,
        labels=labels,
    )

    row = {
        "model_name": MODEL_NAME,
        "fold": fold,
        "n_train": len(train_idx),
        "n_valid": len(valid_idx),
        "n_pc_used": n_components,
        "n_removed": len(remove_indices),
        "removed_pcs": ",".join(removed_pcs),

        "raw_qwk": raw_metrics["qwk"],
        "raw_accuracy": raw_metrics["accuracy"],
        "raw_macro_f1": raw_metrics["macro_f1"],

        "clean_qwk": clean_metrics["qwk"],
        "clean_accuracy": clean_metrics["accuracy"],
        "clean_macro_f1": clean_metrics["macro_f1"],
    }

    row["delta_qwk"] = row["clean_qwk"] - row["raw_qwk"]
    row["delta_accuracy"] = row["clean_accuracy"] - row["raw_accuracy"]
    row["delta_macro_f1"] = row["clean_macro_f1"] - row["raw_macro_f1"]

    fold_rows.append(row)

fold_results_df = pd.DataFrame(fold_rows)
selection_all_df = pd.concat(selection_dfs, ignore_index=True)

# =========================
# 4. Results
# =========================

print("\n===== Fold results =====")
display(fold_results_df)

metric_cols = [
    "raw_qwk",
    "clean_qwk",
    "delta_qwk",
    "raw_accuracy",
    "clean_accuracy",
    "delta_accuracy",
    "raw_macro_f1",
    "clean_macro_f1",
    "delta_macro_f1",
    "n_removed",
]

summary_df = fold_results_df[metric_cols].agg(["mean", "std"]).T

print("\n===== Summary mean/std =====")
display(summary_df)

removed_detail_df = selection_all_df[selection_all_df["remove"]].copy()

print("\n===== Removed PC detail =====")
display(
    removed_detail_df[
        [
            "fold",
            "PC",
            "disease_score_abs_spearman",
            "disease_spearman_rho",
            "max_nuisance_score",
            "max_nuisance_factor",
            "max_nuisance_type",
            "max_nuisance_raw_metric",
        ]
    ].sort_values(["fold", "PC"])
)

# 保存结果
fold_results_path = os.path.join(OUT_DIR, f"{MODEL_NAME}_kfold_performance.csv")
selection_path = os.path.join(OUT_DIR, f"{MODEL_NAME}_pc_selection_detail.csv")

fold_results_df.to_csv(fold_results_path, index=False, encoding="utf-8-sig")
selection_all_df.to_csv(selection_path, index=False, encoding="utf-8-sig")

print(f"\nSaved fold results to: {fold_results_path}")
print(f"Saved PC selection detail to: {selection_path}")

Model: FVs_medsiglip
X shape after filtering: (1189, 1152)
Label distribution:
0    353
1    238
2    238
3    240
4    120
Name: count, dtype: int64

===== Fold 1/5 =====
train=951, valid=238
Removed PCs: ['PC2', 'PC3', 'PC5', 'PC6']

===== Fold 2/5 =====
train=951, valid=238
Removed PCs: ['PC2', 'PC3', 'PC5']

===== Fold 3/5 =====
train=951, valid=238
Removed PCs: ['PC2', 'PC3', 'PC5', 'PC7']

===== Fold 4/5 =====
train=951, valid=238
Removed PCs: ['PC2', 'PC3', 'PC5']

===== Fold 5/5 =====
train=952, valid=237
Removed PCs: ['PC2', 'PC3', 'PC5']

===== Fold results =====


,model_name,fold,n_train,n_valid,n_pc_used,n_removed,removed_pcs,raw_qwk,raw_accuracy,raw_macro_f1,clean_qwk,clean_accuracy,clean_macro_f1,delta_qwk,delta_accuracy,delta_macro_f1
0,FVs_medsiglip,1,951,238,50,4,"PC2,PC3,PC5,PC6",0.830271,0.739496,0.721603,0.802135,0.701681,0.674524,-0.028135,-0.037815,-0.047079
1,FVs_medsiglip,2,951,238,50,3,"PC2,PC3,PC5",0.877935,0.798319,0.775693,0.900042,0.810924,0.788365,0.022107,0.012605,0.012673
2,FVs_medsiglip,3,951,238,50,4,"PC2,PC3,PC5,PC7",0.893331,0.764706,0.732802,0.844220,0.726891,0.704738,-0.049110,-0.037815,-0.028064
3,FVs_medsiglip,4,951,238,50,3,"PC2,PC3,PC5",0.886276,0.785714,0.746629,0.875713,0.760504,0.724271,-0.010562,-0.025210,-0.022358
4,FVs_medsiglip,5,952,237,50,3,"PC2,PC3,PC5",0.840908,0.746835,0.696417,0.833328,0.746835,0.696859,-0.007579,0.000000,0.000442



===== Summary mean/std =====


,mean,std
raw_qwk,0.865744,0.028312
clean_qwk,0.851088,0.037953
delta_qwk,-0.014656,0.026397
raw_accuracy,0.767014,0.024997
clean_accuracy,0.749367,0.040923
delta_accuracy,-0.017647,0.022898
raw_macro_f1,0.734629,0.029425
clean_macro_f1,0.717752,0.043311
delta_macro_f1,-0.016877,0.023667
n_removed,3.400000,0.547723



===== Removed PC detail =====


,fold,PC,disease_score_abs_spearman,disease_spearman_rho,max_nuisance_score,max_nuisance_factor,max_nuisance_type,max_nuisance_raw_metric
1,1,PC2,0.042120,0.042120,0.917426,eye,binary_auc,0.958713
2,1,PC3,0.122897,-0.122897,0.480968,Clarity,spearman,0.480968
4,1,PC5,0.069236,0.069236,0.593441,view_no,binary_auc,0.796721
5,1,PC6,0.105584,0.105584,0.448029,view_no,binary_auc,0.275985
51,2,PC2,0.001388,0.001388,0.918735,eye,binary_auc,0.959368
52,2,PC3,0.126977,-0.126977,0.439653,Clarity,spearman,0.439653
54,2,PC5,0.050819,-0.050819,0.635153,view_no,binary_auc,0.182424
101,3,PC2,0.029508,0.029508,0.924645,eye,binary_auc,0.962323
102,3,PC3,0.113671,-0.113671,0.483077,Clarity,spearman,0.483077
104,3,PC5,0.086437,-0.086437,0.578643,view_no,binary_auc,0.210679



Saved fold results to: ./kfold_pc_removal_one_model\FVs_medsiglip_kfold_performance.csv
Saved PC selection detail to: ./kfold_pc_removal_one_model\FVs_medsiglip_pc_selection_detail.csv


In [10]:
removed_summary = (
    selection_all_df[selection_all_df["remove"]]
    .groupby("PC")
    .agg(
        removed_count=("remove", "count"),
        mean_disease_score=("disease_score_abs_spearman", "mean"),
        mean_nuisance_score=("max_nuisance_score", "mean"),
        main_nuisance_factor=("max_nuisance_factor", lambda x: x.value_counts().index[0]),
    )
    .reset_index()
    .sort_values(["removed_count", "mean_nuisance_score"], ascending=False)
)

display(removed_summary)

,PC,removed_count,mean_disease_score,mean_nuisance_score,main_nuisance_factor
0,PC2,5,0.021710,0.922959,eye
2,PC5,5,0.071505,0.618597,view_no
1,PC3,5,0.125241,0.466644,Clarity
4,PC7,1,0.078864,0.449637,view_no
3,PC6,1,0.105584,0.448029,view_no
